# Road PCI Intelligence — Stage 3D: Asymmetric Multi-Seed Training & Cross-Domain RDD Evaluation

This notebook executes:
1. **Asymmetric Multi-Seed Training** (Seeds 42, 43, 44; 30 epochs each):
   - Crack channel: supervised on crack500 and road_crack; masked out on pothole images.
   - Pothole channel: supervised on pothole images and supervised as all-zero negative targets on crack500 and road_crack.
   - Checkpoints: `last.pt` saved only on final epoch (epoch 30) for each seed in `/kaggle/working/asymmetric_seed<seed>/last.pt`.
   - Evaluation: Best epoch and final epoch 30 metrics, road-crack non-empty Dice, empty clean count, Mean +/- Std (both sample and population std).
2. **Cross-Domain RDD Evaluation** (RDD Validation Split Only, N = 2,560 images):
   - Compares Model A (Masked Baseline), Model B (Naive Baseline), and Model C (Asymmetric Seed 42, Epoch 30).
   - Full image padded to multiple of 32, sigmoid activation, threshold 0.5.
   - Uses exact class names from `classes.txt`.
   - Combined A / B / C evaluation table and 12-image overlay figure with 4 columns: `Image | Model A | Model B | Model C`.


In [ ]:
# ==============================================================================
# 1. ENVIRONMENT & REPOSITORY SETUP
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.5.0', 'albumentations==2.0.8'], check=True)

repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print('Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/skilloutofbounds/road-pci-intelligence.git', str(repo_dir)], check=True)
else:
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', str(repo_dir), 'pull'], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
from src.train import seed_everything, log_library_versions, build_model, run_full_training
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.datasets import RoadDistressDataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np

seed_everything(42)
log_library_versions()
assert torch.cuda.is_available(), 'CRITICAL: CUDA GPU is required!'
device = torch.device('cuda')
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & LOAD SEGMENTATION MANIFEST
# ==============================================================================
dataset_roots = {}
for src_key, spec in DATASET_SPECS.items():
    root = resolve_dataset_root(spec['owner'], spec['name'])
    assert root is not None, f'Dataset {src_key} ({spec["name"]}) root not found!'
    dataset_roots[src_key] = root
    print(f'  • {src_key:12s}: {root}')

manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
assert manifest_path.exists(), f'Missing segmentation manifest: {manifest_path}'
df_seg = pd.read_csv(manifest_path)
print(f'Loaded segmentation manifest: {len(df_seg):,} records.')


In [ ]:
# ==============================================================================
# 3. TRAIN ASYMMETRIC SEEDS 42, 43, 44 (30 EPOCHS EACH)
# ==============================================================================
import time
import json
import cv2
from torch.cuda.amp import autocast
from src.metrics import DistressMetricTracker

seeds = [42, 43, 44]
seed_records = {}

t_all_start = time.time()

# Validation dataset for standalone road_crack image-level metrics
val_dataset = RoadDistressDataset(manifest_df=df_seg, dataset_roots=dataset_roots, split='val')
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

def evaluate_road_crack_details(model_state):
    eval_model = build_model(device)
    eval_model.load_state_dict({k: v.to(device) for k, v in model_state.items()})
    eval_model.eval()
    
    rc_dices_non_empty = []
    empty_clean_count = 0
    
    with torch.no_grad():
        for batch in val_loader:
            source = batch['source'][0]
            if source != 'road_crack':
                continue
            img = batch['image'].to(device)
            target = batch['target'][:, 0, :, :].numpy()[0]
            
            _, _, h, w = img.shape
            pad_h = int(np.ceil(h / 32.0) * 32) - h
            pad_w = int(np.ceil(w / 32.0) * 32) - w
            if pad_h > 0 or pad_w > 0:
                img = torch.nn.functional.pad(img, (0, pad_w, 0, pad_h), mode='constant', value=0)
            
            with autocast():
                logits = eval_model(img)
                prob = torch.sigmoid(logits)[0, 0, :h, :w].cpu().numpy()
            
            pred = (prob >= 0.5).astype(np.uint8)
            gt = target.astype(np.uint8)
            
            gt_fg = (gt == 1).sum()
            pred_fg = (pred == 1).sum()
            
            if gt_fg > 0:
                inter = np.logical_and(pred == 1, gt == 1).sum()
                dice = float((2.0 * inter + 1e-6) / (pred_fg + gt_fg + 1e-6))
                rc_dices_non_empty.append(dice)
            else:
                if pred_fg == 0:
                    empty_clean_count += 1
                    
    return float(np.mean(rc_dices_non_empty)), empty_clean_count

for s in seeds:
    print('\n' + '#' * 80)
    print(f'STARTING ASYMMETRIC TRAINING: Seed = {s}')
    print('#' * 80)
    
    out_d = Path(f'/kaggle/working/asymmetric_seed{s}')
    out_d.mkdir(parents=True, exist_ok=True)
    
    t0_s = time.time()
    model, history, sweep = run_full_training(
        manifest_df=df_seg,
        dataset_roots=dataset_roots,
        epochs=30,
        loss_mode='asymmetric',
        loss_terms='bce+dice',
        seed=s,
        output_dir=out_d,
        save_weights=False,
        save_last_only=True
    )
    t_elapsed_s = time.time() - t0_s
    print(f'[COMPLETED] asymmetric_seed{s} in {t_elapsed_s:.2f}s ({t_elapsed_s/60:.2f} min)')
    
    # Find best epoch and final epoch records
    best_rec = max(history, key=lambda x: x['selection_score'])
    final_rec = history[-1]
    
    best_ep = best_rec['epoch']
    
    # Final epoch weights are in out_d / 'last.pt'
    last_weights = torch.load(out_d / 'last.pt', map_location='cpu')
    rc_non_empty_final, rc_clean_final = evaluate_road_crack_details(last_weights)
    
    # For best epoch road crack details: if best_ep == 30, it is identical; else evaluate best weights
    if best_ep == 30:
        rc_non_empty_best, rc_clean_best = rc_non_empty_final, rc_clean_final
    else:
        # Load threshold sweep results at 0.5
        # The model variable currently holds best model weights from threshold sweep in run_full_training!
        rc_non_empty_best, rc_clean_best = evaluate_road_crack_details(model.state_dict())
        
    seed_records[s] = {
        'best_epoch': {
            'epoch': best_ep,
            'selection_score': float(best_rec['selection_score']),
            'c500_dice': float(best_rec['val_metrics']['crack500_dataset_dice']),
            'pot_dice': float(best_rec['val_metrics']['pothole_dataset_dice']),
            'rc_dice': float(best_rec['val_metrics']['road_crack_dataset_dice']),
            'rc_non_empty_dice': rc_non_empty_best,
            'rc_empty_clean': rc_clean_best
        },
        'final_epoch': {
            'epoch': 30,
            'selection_score': float(final_rec['selection_score']),
            'c500_dice': float(final_rec['val_metrics']['crack500_dataset_dice']),
            'pot_dice': float(final_rec['val_metrics']['pothole_dataset_dice']),
            'rc_dice': float(final_rec['val_metrics']['road_crack_dataset_dice']),
            'rc_non_empty_dice': rc_non_empty_final,
            'rc_empty_clean': rc_clean_final
        }
    }

print(f'\nAll 3 asymmetric seed runs completed in {time.time() - t_all_start:.2f}s.')


In [ ]:
# ==============================================================================
# 4. REPORT MULTI-SEED BENCHMARK TABLES (BEST & FINAL EPOCH 30)
# ==============================================================================
print('=' * 115)
print('MULTI-SEED EVALUATION TABLE — BEST EPOCH SELECTION (Threshold 0.5, Asymmetric Mode)')
print('=' * 115)
print(f"{'Mode':<10} | {'Seed':<5} | {'Best Ep':<7} | {'Select Score':<12} | {'CRACK500 Dice':<14} | {'Pothole Dice':<13} | {'RoadCrack Dice':<15} | {'RC NonEmpty':<12} | {'RC Empty 0-Pred':<16}")
print('-' * 115)

for s in seeds:
    r = seed_records[s]['best_epoch']
    print(f"{'asymmetric':<10} | {s:<5d} | {r['epoch']:<7d} | {r['selection_score']:<12.4f} | {r['c500_dice']:<14.4f} | {r['pot_dice']:<13.4f} | {r['rc_dice']:<15.4f} | {r['rc_non_empty_dice']:<12.4f} | {r['rc_empty_clean']:2d}/39 ({r['rc_empty_clean']/39*100:4.1f}%)")
print('=' * 115 + '\n')

print('=' * 115)
print('MULTI-SEED EVALUATION TABLE — FINAL EPOCH 30 (Threshold 0.5, Asymmetric Mode)')
print('=' * 115)
print(f"{'Mode':<10} | {'Seed':<5} | {'Epoch':<7} | {'Select Score':<12} | {'CRACK500 Dice':<14} | {'Pothole Dice':<13} | {'RoadCrack Dice':<15} | {'RC NonEmpty':<12} | {'RC Empty 0-Pred':<16}")
print('-' * 115)

for s in seeds:
    r = seed_records[s]['final_epoch']
    print(f"{'asymmetric':<10} | {s:<5d} | {r['epoch']:<7d} | {r['selection_score']:<12.4f} | {r['c500_dice']:<14.4f} | {r['pot_dice']:<13.4f} | {r['rc_dice']:<15.4f} | {r['rc_non_empty_dice']:<12.4f} | {r['rc_empty_clean']:2d}/39 ({r['rc_empty_clean']/39*100:4.1f}%)")
print('=' * 115 + '\n')

# Compute Mean +/- Std (both Sample Std ddof=1 and Population Std ddof=0)
print('=' * 115)
print('SUMMARY STATISTICS ACROSS 3 SEEDS (42, 43, 44) — ASYMMETRIC LOSS')
print('=' * 115)
print(f"{'Metric':<25} | {'Best Epoch (Sample Std, ddof=1)':<32} | {'Final Epoch 30 (Sample Std, ddof=1)':<34} | {'Best (Pop Std, ddof=0)':<23} | {'Final (Pop Std, ddof=0)':<23}")
print('-' * 115)

keys = [
    ('Selection Score', 'selection_score'),
    ('CRACK500 Dice', 'c500_dice'),
    ('Pothole Dice', 'pot_dice'),
    ('Road Crack Dice', 'rc_dice'),
    ('Road Crack Non-Empty', 'rc_non_empty_dice'),
    ('RC Empty Clean (out of 39)', 'rc_empty_clean')
]

for label, k in keys:
    vals_best = [seed_records[s]['best_epoch'][k] for s in seeds]
    vals_final = [seed_records[s]['final_epoch'][k] for s in seeds]
    
    m_b = np.mean(vals_best)
    s_b_sample = np.std(vals_best, ddof=1)
    s_b_pop = np.std(vals_best, ddof=0)
    
    m_f = np.mean(vals_final)
    s_f_sample = np.std(vals_final, ddof=1)
    s_f_pop = np.std(vals_final, ddof=0)
    
    print(f"{label:<25} | {m_b:7.4f} +/- {s_b_sample:7.4f}              | {m_f:7.4f} +/- {s_f_sample:7.4f}                | {m_b:7.4f} +/- {s_b_pop:7.4f}       | {m_f:7.4f} +/- {s_f_pop:7.4f}")
print('=' * 115 + '\n')

with open('/kaggle/working/asymmetric_seed_results.json', 'w', encoding='utf-8') as f:
    json.dump(seed_records, f, indent=2)
print('[OK] Saved asymmetric_seed_results.json')


In [ ]:
# ==============================================================================
# 5. CROSS-DOMAIN RDD EVALUATION (MODEL A vs MODEL B vs MODEL C [EPOCH 30])
# ==============================================================================
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import random

# Locate Model A best.pt
k_input = Path('/kaggle/input')
cand_a = list(k_input.rglob('*run_a*/**/best.pt')) + list(k_input.rglob('*masked*/**/best.pt'))
assert len(cand_a) > 0, f'Could not find Model A best.pt: {cand_a}'
model_a_pt = cand_a[0]

# Locate Model B best.pt
cand_b = list(k_input.rglob('*run_b*/**/best.pt')) + list(k_input.rglob('*naive*/**/best.pt'))
assert len(cand_b) > 0, f'Could not find Model B best.pt: {cand_b}'
model_b_pt = cand_b[0]

# Locate Model C last.pt (Epoch 30 of asymmetric seed 42)
model_c_pt = Path('/kaggle/working/asymmetric_seed42/last.pt')
assert model_c_pt.exists(), f'Missing Model C last.pt at {model_c_pt}'

print(f'Model A Checkpoint: {model_a_pt}')
print(f'Model B Checkpoint: {model_b_pt}')
print(f'Model C Checkpoint: {model_c_pt}')

def load_eval_model(weights_p):
    m = build_model(device)
    st = torch.load(weights_p, map_location=device)
    m.load_state_dict(st)
    m.eval()
    return m

model_a = load_eval_model(model_a_pt)
model_b = load_eval_model(model_b_pt)
model_c = load_eval_model(model_c_pt)
print('[OK] Loaded Models A, B, and C successfully.')

# Load RDD manifest strictly for VAL split
manifest_rdd_p = repo_dir / 'manifests/rdd2022es_manifest.csv'
df_rdd = pd.read_csv(manifest_rdd_p)
df_rdd_val = df_rdd[df_rdd['split'] == 'val'].copy().reset_index(drop=True)
print(f'Filtered to RDD VAL split: {len(df_rdd_val):,} images (Target: 2,560)')
assert len(df_rdd_val) == 2560

rdd_root = dataset_roots['rdd2022es']

RDD_CLASSES = {
    0: 'longitudinal',
    1: 'longitudinal wide',
    2: 'transverse',
    3: 'transverse wide',
    4: 'alligator',
    5: 'alligator sunken',
    6: 'pothole',
    7: 'pothole deep'
}

def parse_rdd_boxes(txt_path, width, height):
    boxes = []
    if not txt_path.exists():
        return boxes
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                xc = float(parts[1]) * width
                yc = float(parts[2]) * height
                bw = float(parts[3]) * width
                bh = float(parts[4]) * height
                xmin = max(0, int(round(xc - bw / 2.0)))
                ymin = max(0, int(round(yc - bh / 2.0)))
                xmax = min(width, int(round(xc + bw / 2.0)))
                ymax = min(height, int(round(yc + bh / 2.0)))
                if xmax > xmin and ymax > ymin:
                    boxes.append({
                        'class_id': cid,
                        'box': (ymin, ymax, xmin, xmax),
                        'area': (ymax - ymin) * (xmax - xmin)
                    })
    return boxes

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

@torch.no_grad()
def infer_models_padded(img_rgb):
    h, w = img_rgb.shape[:2]
    pad_h = int(np.ceil(h / 32.0) * 32) - h
    pad_w = int(np.ceil(w / 32.0) * 32) - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
    norm = ((img_padded / 255.0) - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
    
    with autocast():
        p_a = torch.sigmoid(model_a(tensor))[0, :, :h, :w].cpu().numpy()
        p_b = torch.sigmoid(model_b(tensor))[0, :, :h, :w].cpu().numpy()
        p_c = torch.sigmoid(model_c(tensor))[0, :, :h, :w].cpu().numpy()
    return p_a, p_b, p_c

print('Running RDD inference across 2,560 validation images for Models A, B, and C...')
t0_eval = time.time()
rdd_records = []

for idx, row in df_rdd_val.iterrows():
    img_p = rdd_root / row['image_path']
    txt_p = rdd_root / row['mask_path_or_label_path'] if row['mask_path_or_label_path'] else Path('')
    
    with Image.open(img_p) as pil:
        img_rgb = np.array(pil.convert('RGB'))
    h, w = img_rgb.shape[:2]
    n_pixels = h * w
    
    boxes = parse_rdd_boxes(txt_p, w, h)
    crack_boxes = [b for b in boxes if 0 <= b['class_id'] <= 5]
    pothole_boxes = [b for b in boxes if 6 <= b['class_id'] <= 7]
    
    pa, pb, pc = infer_models_padded(img_rgb)
    
    pred_a_crk = (pa[0] >= 0.5)
    pred_a_pot = (pa[1] >= 0.5)
    pred_b_crk = (pb[0] >= 0.5)
    pred_b_pot = (pb[1] >= 0.5)
    pred_c_crk = (pc[0] >= 0.5)
    pred_c_pot = (pc[1] >= 0.5)
    
    rdd_records.append({
        'index': idx,
        'image_path': str(img_p),
        'boxes': boxes,
        'crack_boxes': crack_boxes,
        'pothole_boxes': pothole_boxes,
        'n_pixels': n_pixels,
        'a_crack_fg_frac': float(pred_a_crk.sum() / n_pixels),
        'a_pothole_fg_frac': float(pred_a_pot.sum() / n_pixels),
        'a_pred_crack': pred_a_crk,
        'a_pred_pothole': pred_a_pot,
        'b_crack_fg_frac': float(pred_b_crk.sum() / n_pixels),
        'b_pothole_fg_frac': float(pred_b_pot.sum() / n_pixels),
        'b_pred_crack': pred_b_crk,
        'b_pred_pothole': pred_b_pot,
        'c_crack_fg_frac': float(pred_c_crk.sum() / n_pixels),
        'c_pothole_fg_frac': float(pred_c_pot.sum() / n_pixels),
        'c_pred_crack': pred_c_crk,
        'c_pred_pothole': pred_c_pot,
    })
    if (idx + 1) % 500 == 0 or (idx + 1) == len(df_rdd_val):
        print(f'Processed {idx + 1:4d} / {len(df_rdd_val)} images ({time.time() - t0_eval:.1f}s)...')

print(f'Completed RDD cross-domain evaluation in {time.time() - t0_eval:.2f}s.')


In [ ]:
# ==============================================================================
# 6. PRINT COMBINED A / B / C RDD EVALUATION TABLES
# ==============================================================================
# Part A: Zero-Box Images
zero_records = [r for r in rdd_records if len(r['boxes']) == 0]
n_z = len(zero_records)
print('=' * 105)
print(f'A. IMAGES WITH ZERO BOXES (N = {n_z:,} images)')
print('=' * 105)
print(f"{'Model':<22} | {'Channel':<9} | {'Flag > 0.1%':<16} | {'Flag > 0.5%':<16} | {'Flag > 1.0%':<16} | {'Mean FG Frac':<14}")
print('-' * 105)

models = [('a', 'Model A (Masked)'), ('b', 'Model B (Naive)'), ('c', 'Model C (Asymmetric, Ep30)')]

for m_key, m_name in models:
    for ch in ['crack', 'pothole']:
        fracs = np.array([r[f'{m_key}_{ch}_fg_frac'] for r in zero_records])
        r01 = np.mean(fracs > 0.001) * 100
        r05 = np.mean(fracs > 0.005) * 100
        r10 = np.mean(fracs > 0.010) * 100
        m_frac = np.mean(fracs) * 100
        print(f"{m_name:<22} | {ch:<9} | {r01:6.2f}% ({np.sum(fracs > 0.001):4d}) | {r05:6.2f}% ({np.sum(fracs > 0.005):4d}) | {r10:6.2f}% ({np.sum(fracs > 0.010):4d}) | {m_frac:6.4f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Part B: Hit Rate & Box Pixel Coverage
def compute_box_metrics(target_records, box_key, pred_key, cid_filter=None):
    img_hits = []
    box_fracs = []
    for r in target_records:
        boxes = [b for b in r[box_key] if cid_filter is None or b['class_id'] == cid_filter]
        if not boxes:
            continue
        pred_mask = r[pred_key]
        img_has_hit = False
        for b in boxes:
            ymin, ymax, xmin, xmax = b['box']
            area = b['area']
            p_in = pred_mask[ymin:ymax, xmin:xmax].sum()
            frac = float(p_in / area)
            box_fracs.append(frac)
            if frac >= 0.05:
                img_has_hit = True
        img_hits.append(img_has_hit)
    n_i = len(img_hits)
    hr = np.mean(img_hits) * 100 if n_i > 0 else 0.0
    mbf = np.mean(box_fracs) * 100 if box_fracs else 0.0
    return n_i, len(box_fracs), hr, mbf

# Pothole Channel vs Pothole Boxes
pot_records = [r for r in rdd_records if len(r['pothole_boxes']) > 0]
print('=' * 105)
print(f'B. POTHOLE CHANNEL vs POTHOLE BOXES (N = {len(pot_records)} images, 498 boxes)')
print('=' * 105)
print(f"{'Model':<24} | {'Category':<24} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    n_i, n_b, hr, mbf = compute_box_metrics(pot_records, 'pothole_boxes', f'{m_key}_pred_pothole')
    print(f"{m_name:<24} | {'All Potholes (6-7)':<24} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in [6, 7]:
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(pot_records, 'pothole_boxes', f'{m_key}_pred_pothole', cid_filter=cid)
        c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
        print(f"{m_name:<24} | {c_label:<24} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Crack Channel vs Crack Boxes
crk_records = [r for r in rdd_records if len(r['crack_boxes']) > 0]
print('=' * 105)
print(f'CRACK CHANNEL vs CRACK BOXES (N = {len(crk_records)} images, 3,278 boxes)')
print('=' * 105)
print(f"{'Model':<24} | {'Category':<24} | {'Images':<8} | {'Boxes':<8} | {'Hit Rate (>=5%)':<18} | {'Mean Box Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    n_i, n_b, hr, mbf = compute_box_metrics(crk_records, 'crack_boxes', f'{m_key}_pred_crack')
    print(f"{m_name:<24} | {'All Cracks (0-5)':<24} | {n_i:<8} | {n_b:<8} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
    for cid in range(6):
        n_ic, n_bc, hrc, mbfc = compute_box_metrics(crk_records, 'crack_boxes', f'{m_key}_pred_crack', cid_filter=cid)
        c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
        print(f"{m_name:<24} | {c_label:<24} | {n_ic:<8} | {n_bc:<8} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')

# Part C: Cross-Channel Audit
only_crk = [r for r in rdd_records if len(r['crack_boxes']) > 0 and len(r['pothole_boxes']) == 0]
only_pot = [r for r in rdd_records if len(r['pothole_boxes']) > 0 and len(r['crack_boxes']) == 0]
print('=' * 105)
print('C. CROSS-CHANNEL AUDIT')
print(f'  • Images with ONLY Crack boxes:   {len(only_crk):,} images')
print(f'  • Images with ONLY Pothole boxes: {len(only_pot):,} images')
print('=' * 105)
print(f"{'Model':<24} | {'Condition':<25} | {'Evaluated Ch':<14} | {'Flag > 0.1%':<16} | {'Flag > 0.5%':<16} | {'Mean FG Frac':<14}")
print('-' * 105)

for m_key, m_name in models:
    pot_fracs = np.array([r[f'{m_key}_pothole_fg_frac'] for r in only_crk])
    r01_p = np.mean(pot_fracs > 0.001) * 100
    r05_p = np.mean(pot_fracs > 0.005) * 100
    m_p = np.mean(pot_fracs) * 100
    print(f"{m_name:<24} | {'Only Crack Boxes':<25} | {'Pothole':<14} | {r01_p:6.2f}% ({np.sum(pot_fracs > 0.001):4d}) | {r05_p:6.2f}% ({np.sum(pot_fracs > 0.005):4d}) | {m_p:6.4f}%")
    
    crk_fracs = np.array([r[f'{m_key}_crack_fg_frac'] for r in only_pot])
    r01_c = np.mean(crk_fracs > 0.001) * 100
    r05_c = np.mean(crk_fracs > 0.005) * 100
    m_c = np.mean(crk_fracs) * 100
    print(f"{m_name:<24} | {'Only Pothole Boxes':<25} | {'Crack':<14} | {r01_c:6.2f}% ({np.sum(crk_fracs > 0.001):4d}) | {r05_c:6.2f}% ({np.sum(crk_fracs > 0.005):4d}) | {m_c:6.4f}%")
    if m_key != 'c':
        print('-' * 105)
print('=' * 105 + '\n')


In [ ]:
# ==============================================================================
# 7. 12-IMAGE VISUALIZATION (ORIGINAL | MODEL A | MODEL B | MODEL C)
# ==============================================================================
rng = random.Random(42)

clean_cands = [r for r in rdd_records if len(r['boxes']) == 0]
sel_clean = rng.sample(clean_cands, min(4, len(clean_cands)))

pot_cands = [r for r in rdd_records if len(r['pothole_boxes']) > 0]
sel_pot = rng.sample(pot_cands, min(4, len(pot_cands)))

# Classes 0, 1 = longitudinal; Classes 4, 5 = alligator
crk_cands = [r for r in rdd_records if any(b['class_id'] in [0, 1, 4, 5] for b in r['crack_boxes'])]
sel_crk = rng.sample(crk_cands, min(4, len(crk_cands)))

sel_12 = (
    [('Clean', r) for r in sel_clean] +
    [('Pothole', r) for r in sel_pot] +
    [('Crack (Long/Allig)', r) for r in sel_crk]
)

fig, axes = plt.subplots(12, 4, figsize=(20, 36))

for row_idx, (cat_name, r) in enumerate(sel_12):
    with Image.open(r['image_path']) as pil:
        img_rgb = np.array(pil.convert('RGB'))
    
    img_gt = img_rgb.copy()
    for b in r['boxes']:
        ymin, ymax, xmin, xmax = b['box']
        cid = b['class_id']
        color = (0, 100, 255) if cid in [6, 7] else (255, 50, 50)
        cv2.rectangle(img_gt, (xmin, ymin), (xmax, ymax), color, 2)
        c_short = RDD_CLASSES[cid][:6]
        cv2.putText(img_gt, f"{cid}:{c_short}", (xmin, max(15, ymin - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        
    stem = Path(r['image_path']).name
    if len(stem) > 22:
        stem = stem[:19] + '...'
        
    # Col 0: Image + GT Boxes
    axes[row_idx, 0].imshow(img_gt)
    axes[row_idx, 0].set_title(f"[{cat_name}] {stem} ({len(r['boxes'])} boxes)", fontsize=8.5, fontweight='bold')
    axes[row_idx, 0].axis('off')
    
    # Col 1: Model A
    oa = img_rgb.copy()
    oa[r['a_pred_crack']] = [230, 20, 20]
    oa[r['a_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 1].imshow(oa)
    axes[row_idx, 1].set_title(f"Model A (Masked) | Crk:{r['a_crack_fg_frac']*100:.2f}% Pot:{r['a_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 1].axis('off')
    
    # Col 2: Model B
    ob = img_rgb.copy()
    ob[r['b_pred_crack']] = [230, 20, 20]
    ob[r['b_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 2].imshow(ob)
    axes[row_idx, 2].set_title(f"Model B (Naive) | Crk:{r['b_crack_fg_frac']*100:.2f}% Pot:{r['b_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 2].axis('off')
    
    # Col 3: Model C
    oc = img_rgb.copy()
    oc[r['c_pred_crack']] = [230, 20, 20]
    oc[r['c_pred_pothole']] = [20, 100, 240]
    axes[row_idx, 3].imshow(oc)
    axes[row_idx, 3].set_title(f"Model C (Asymmetric) | Crk:{r['c_crack_fg_frac']*100:.2f}% Pot:{r['c_pothole_fg_frac']*100:.2f}%", fontsize=8)
    axes[row_idx, 3].axis('off')

plt.subplots_adjust(top=0.98, bottom=0.01, hspace=0.25, wspace=0.08)
out_fig = Path('/kaggle/working/rdd_cross_domain_12_val_abc.png')
plt.savefig(out_fig, dpi=120, bbox_inches='tight')
plt.close()
print(f'[OK] Saved 12-image 4-column figure to {out_fig}')
